In [21]:
import pandas as pd
import numpy as np
import normalization
import importlib
importlib.reload(normalization)

<module 'normalization' from '/home/sebastian/fall_of_man/even_neuer/germs/normalization.py'>

In [22]:
lines = []
with open('nouns.txt', 'r') as nouns:
    for x in nouns:
        new_line = x.split('\t')
        if len(new_line) > 1:
            new_line[2] = new_line[2][:-1]
            lines.append(new_line)

In [23]:
normalized_lines = normalization.normalize_data(lines)

In [24]:
cleansed_lines = normalization.purge_pure_duplicates(normalized_lines)

In [25]:
chassis = pd.read_csv('nouns.csv')

/tmp/ipykernel_49769/774707866.py:1: DtypeWarning: Columns (5,6,8,11,12,20,21,29,30,38,39,47,48,56,57,65,66,74,75) have mixed types. Specify dtype option on import or set low_memory=False.
  chassis = pd.read_csv('nouns.csv')


In [26]:
chassis.columns

Index(['lemma', 'pos', 'genus', 'genus 1', 'genus 2', 'genus 3', 'genus 4',
       'nominativ singular', 'nominativ singular*', 'nominativ singular 1',
       'nominativ singular 2', 'nominativ singular 3', 'nominativ singular 4',
       'nominativ singular stark', 'nominativ singular schwach',
       'nominativ singular gemischt', 'nominativ plural', 'nominativ plural*',
       'nominativ plural 1', 'nominativ plural 2', 'nominativ plural 3',
       'nominativ plural 4', 'nominativ plural stark',
       'nominativ plural schwach', 'nominativ plural gemischt',
       'genitiv singular', 'genitiv singular*', 'genitiv singular 1',
       'genitiv singular 2', 'genitiv singular 3', 'genitiv singular 4',
       'genitiv singular stark', 'genitiv singular schwach',
       'genitiv singular gemischt', 'genitiv plural', 'genitiv plural*',
       'genitiv plural 1', 'genitiv plural 2', 'genitiv plural 3',
       'genitiv plural 4', 'genitiv plural stark', 'genitiv plural schwach',
       'geni

In [32]:
cleansed_lines = sorted(cleansed_lines, key=lambda x: x[2])
print(cleansed_lines)

[['170.', 'Dinner', 'Das Abendessen', 'Die Abendessen'], ['1286.', 'Adventure', 'Das Abenteuer', 'Die Abenteuer'], ['1773.', 'Compartment', 'Das Abteil', 'Die Abteile'], ['2086.', 'Badge', 'Das Abzeichen', 'Die Abzeichen'], ['1698.', 'Shrug', 'Das Achselzucken', '-'], ['2668.', 'Gentry', 'Das Adel', '-'], ['2153.', 'Album', 'Das Album', 'Die Alben'], ['743.', 'Alien', 'Das Alien', 'Die Aliens'], ['173.', 'Age', 'Das Alter', '-'], ['2420.', 'Aluminum', 'Das Aluminium', '-'], ['1175.', 'Offer', 'Das Angebot', 'Die Angebote'], ['1118.', 'Fishing', 'Das Angeln', '-'], ['2211.', 'Dressing', 'Das Ankleiden', '-'], ['2834.', 'Swell', 'Das Anschwellen', 'Die Anschwellen'], ['1082.', 'Estate', 'Das Anwesen', 'Die Anwesen'], ['936.', 'Argument', 'Das Argument', 'Die Argumente'], ['2142.', 'Bracelet', 'Das Armband', 'Die Armbänder'], ['2864.', 'Aroma', 'Das Aroma', 'Die Aromen'], ['2568.', 'Array', 'Das Array', 'Die Arrays'], ['1505.', 'Asshole', 'Das Arschloch', 'Die Arschlöcher'], ['2364.', 'Ar

In [33]:
mapping = {'Der':'m', 'Die':'f', 'Das':'n'}

df = pd.DataFrame(cleansed_lines, columns=('cardinal', 'english', 'g_singular', 'g_plural'))
df['g_plural_nude'] = df['g_plural'].apply(lambda x: x[4:])
df['g_singular_nude'] = df['g_singular'].apply(lambda x: x[4:] if x[:3] in {'Der', 'Die', 'Das'} else x)
df['sex'] = df['g_singular'].apply(lambda x: mapping[x[:3]] if x[:3] in {'Der', 'Die', 'Das'} else np.nan)
df

,cardinal,english,g_singular,g_plural,g_plural_nude,g_singular_nude,sex
0,170.,Dinner,Das Abendessen,Die Abendessen,Abendessen,Abendessen,n
1,1286.,Adventure,Das Abenteuer,Die Abenteuer,Abenteuer,Abenteuer,n
2,1773.,Compartment,Das Abteil,Die Abteile,Abteile,Abteil,n
3,2086.,Badge,Das Abzeichen,Die Abzeichen,Abzeichen,Abzeichen,n
4,1698.,Shrug,Das Achselzucken,-,,Achselzucken,n
...,...,...,...,...,...,...,...
2549,2869.,Transfer,Die Überweisung,Die Überweisungen,Überweisungen,Überweisung,f
2550,2222.,Conviction,Die Überzeugung,Die Überzeugungen,Überzeugungen,Überzeugung,f
2551,627.,Practice,Die Übung,Die Übungen,Übungen,Übung,f
2552,309.,Mama,Mama,-,,Mama,NaN


In [34]:
result = df.merge(chassis, how='left', left_on=['g_singular_nude', 'sex'], right_on=['nominativ singular', 'genus'])

In [35]:
result['plural_agreement'] = result.apply(lambda x: x['nominativ plural'] == x['g_plural_nude'] or (x['nominativ plural'] is np.nan and x['g_plural_nude'] == ''), axis=1)

In [36]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(result[result['plural_agreement'] == False][['g_singular', 'g_plural_nude', 'nominativ plural']])
print(len(result[result['plural_agreement'] == False]))

                   g_singular          g_plural_nude    nominativ plural
7                   Das Alien                 Aliens                 NaN
8                   Das Alter                                      Alter
13            Das Anschwellen            Anschwellen                 NaN
17                  Das Aroma                 Aromen                 NaN
18                  Das Array                 Arrays                 NaN
24             Das Aufblitzen             Aufblitzen                 NaN
25     Das Aufeinandertreffen     Aufeinandertreffen                 NaN
32                 Das Backup                Backups                 NaN
35              Das Baldachin             Baldachine                 NaN
36                   Das Band                 Bänder                 NaN
37                   Das Band                 Bänder                 NaN
48            Das Bewusstsein                               Bewusstseine
56                   Das Blut                      

In [37]:
chassis.columns

Index(['lemma', 'pos', 'genus', 'genus 1', 'genus 2', 'genus 3', 'genus 4',
       'nominativ singular', 'nominativ singular*', 'nominativ singular 1',
       'nominativ singular 2', 'nominativ singular 3', 'nominativ singular 4',
       'nominativ singular stark', 'nominativ singular schwach',
       'nominativ singular gemischt', 'nominativ plural', 'nominativ plural*',
       'nominativ plural 1', 'nominativ plural 2', 'nominativ plural 3',
       'nominativ plural 4', 'nominativ plural stark',
       'nominativ plural schwach', 'nominativ plural gemischt',
       'genitiv singular', 'genitiv singular*', 'genitiv singular 1',
       'genitiv singular 2', 'genitiv singular 3', 'genitiv singular 4',
       'genitiv singular stark', 'genitiv singular schwach',
       'genitiv singular gemischt', 'genitiv plural', 'genitiv plural*',
       'genitiv plural 1', 'genitiv plural 2', 'genitiv plural 3',
       'genitiv plural 4', 'genitiv plural stark', 'genitiv plural schwach',
       'geni

In [47]:
chassis

,lemma,pos,genus,genus 1,genus 2,genus 3,genus 4,nominativ singular,nominativ singular*,nominativ singular 1,...,akkusativ singular gemischt,akkusativ plural,akkusativ plural*,akkusativ plural 1,akkusativ plural 2,akkusativ plural 3,akkusativ plural 4,akkusativ plural stark,akkusativ plural schwach,akkusativ plural gemischt
0,-algie,"Gebundenes Lexem,Substantiv",f,NaN,NaN,NaN,NaN,-algie,NaN,NaN,...,NaN,-algien,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-ant,"Affix,Substantiv,Suffix",NaN,m,n,NaN,NaN,NaN,NaN,-ant,...,NaN,NaN,NaN,-anten,-ante,NaN,NaN,NaN,NaN,NaN
2,-anz,"Affix,Substantiv,Suffix",f,NaN,NaN,NaN,NaN,-anz,NaN,NaN,...,NaN,-anzen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-chen,"Affix,Substantiv,Suffix",n,NaN,NaN,NaN,NaN,-chen,NaN,NaN,...,NaN,-chen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-elchen,"Affix,Substantiv,Suffix",n,NaN,NaN,NaN,NaN,-elchen,NaN,NaN,...,NaN,-elchen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102439,Ḫāriǧit,Substantiv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102440,’Ndrangheta,Substantiv,f,NaN,NaN,NaN,NaN,’Ndrangheta,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102441,’Sgebengu,Substantiv,m,NaN,NaN,NaN,NaN,’Sgebengu,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102442,’s-Gravenhage,"Substantiv,Toponym",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
